# nanotox — Nanoparticle Cytotoxicity ML Benchmark (Colab)

**Self-contained** notebook for predicting nanoparticle cytotoxicity and
benchmarking machine-learning algorithms (Logistic, SVM, k-NN, Random Forest,
Extra-Trees, Gradient Boosting, XGBoost, LightGBM) with nested cross-validation,
a held-out test set, leave-one-material-family-out generalisation and
permutation importance.

Run top-to-bottom. The package source is written to the Colab filesystem in
Section 1, so **no GitHub access is needed**.

> **Publication caveat.** The bundled dataset is a *physicochemically-grounded
> synthetic generator* so the notebook runs with zero setup. It validates the
> pipeline, **not** a biological claim. Section 3 shows how to import real data
> (eNanoMapper, PubChem, ChEMBL, your own CSV, and the omics repositories) —
> use real data before drawing any scientific conclusion.


## 0. Install dependencies

Colab ships numpy/pandas/matplotlib/scikit-learn; this adds the gradient boosters and SHAP.

In [ ]:
# Boards: xgboost & lightgbm are the usual top performers; shap is optional.
!pip -q install xgboost lightgbm shap 2>/dev/null
import sklearn, numpy, pandas, scipy, matplotlib
print("scikit-learn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)


## 1. Write the `nanotox` package into the Colab filesystem

These cells materialise the exact package modules, then put them on the import
path. (This is what makes the notebook standalone.)

In [ ]:
import os
for d in ["nanotox", "nanotox/data", "nanotox/features", "nanotox/models"]:
    os.makedirs(d, exist_ok=True)
print("package dirs created")


In [ ]:
%%writefile nanotox/__init__.py
"""nanotox: a reproducible ML pipeline for nanoparticle cytotoxicity prediction.

The package is organised into small, composable modules:

* :mod:`nanotox.config`    -- central configuration dataclasses.
* :mod:`nanotox.data`      -- dataset acquisition (online import + reproducible
                              physicochemically-grounded fallback generator).
* :mod:`nanotox.features`  -- physicochemistry-aware feature engineering.
* :mod:`nanotox.models`    -- the model zoo + hyper-parameter search spaces.
* :mod:`nanotox.pipeline`  -- end-to-end training / nested cross-validation.
* :mod:`nanotox.evaluate`  -- metrics, comparison tables and plots.
"""

from __future__ import annotations

__version__ = "0.1.0"

__all__ = ["__version__"]


In [ ]:
%%writefile nanotox/config.py
"""Central configuration for the nanotox pipeline.

Keeping every tunable knob in one place makes experiments reproducible and
makes it trivial to serialise the exact configuration that produced a result
(important for publication-grade work).
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any

# Project root = two levels up from this file (src/nanotox/config.py -> repo).
PROJECT_ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

#: Global seed. A single seed threaded through every stochastic component is
#: the cheapest, highest-value reproducibility guarantee a paper can offer.
RANDOM_STATE = 42

#: Name of the binary target column produced by the data layer.
TARGET_COLUMN = "cytotoxic"


@dataclass
class DataConfig:
    """How the dataset is sourced and materialised on disk."""

    # Remote source tried first. If unreachable (offline CI, air-gapped
    # container) the pipeline falls back to a physicochemically-grounded
    # synthetic generator so the whole repo is runnable with zero network.
    remote_url: str | None = None
    cache_path: Path = DATA_DIR / "nanoparticle_cytotoxicity.csv"
    # Size of the synthetic fallback cohort.
    n_synthetic: int = 1500
    # Fraction of rows held out as a final, untouched test set.
    test_size: float = 0.2
    # Viability threshold (%) below which a particle is labelled cytotoxic.
    viability_cutoff: float = 50.0


@dataclass
class CVConfig:
    """Cross-validation / model-selection protocol."""

    outer_folds: int = 5
    inner_folds: int = 3
    n_iter_search: int = 40  # RandomizedSearchCV budget per model
    scoring: str = "roc_auc"


@dataclass
class PipelineConfig:
    """Top-level configuration object."""

    data: DataConfig = field(default_factory=DataConfig)
    cv: CVConfig = field(default_factory=CVConfig)
    random_state: int = RANDOM_STATE
    results_dir: Path = RESULTS_DIR
    # Restrict the model zoo to these keys (None = all available models).
    models: list[str] | None = None

    def to_dict(self) -> dict[str, Any]:
        """JSON-serialisable snapshot (Paths rendered as strings)."""

        def _coerce(obj: Any) -> Any:
            if isinstance(obj, Path):
                return str(obj)
            if isinstance(obj, dict):
                return {k: _coerce(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [_coerce(v) for v in obj]
            return obj

        return _coerce(asdict(self))


In [ ]:
%%writefile nanotox/data/__init__.py
"""Dataset acquisition for nanotox."""

from __future__ import annotations

from .loader import DatasetBundle, load_dataset, make_synthetic_cohort

__all__ = ["DatasetBundle", "load_dataset", "make_synthetic_cohort"]


In [ ]:
%%writefile nanotox/data/loader.py
"""Dataset acquisition with a graceful offline fallback.

Strategy (in order):

1. If ``remote_url`` is configured and reachable, download the CSV and cache it.
2. Else, if a cached CSV already exists on disk, use it.
3. Else, synthesise a physicochemically-grounded cohort so the pipeline is
   fully runnable with no network access.

The synthetic generator is **not** random noise. It encodes well-established
nano-QSAR relationships from the metal-oxide / lipid / carbon / polymer
literature so that a sane model can actually learn signal, while remaining
clearly labelled as synthetic. Replace the loader's ``remote_url`` (or drop a
curated CSV at ``cache_path``) to train on real experimental data without
touching the rest of the pipeline.

Expected real-world sources this schema is compatible with include
eNanoMapper exports, the NIL/S2NANO cytotoxicity compilations and the metal
oxide nano-QSAR datasets used throughout the literature. Column names below
mirror those conventions.
"""

from __future__ import annotations

import io
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from ..config import DataConfig, TARGET_COLUMN

# --------------------------------------------------------------------------- #
# Schema
# --------------------------------------------------------------------------- #

#: Material families covered. Modelling several families at once is what lets
#: us probe the "cross-family generalisation" gap (train on some, test on a
#: held-out family -- see pipeline.leave_one_family_out).
MATERIAL_FAMILIES = ("metal_oxide", "carbon", "lipid", "polymer")

#: Numeric physicochemical + exposure descriptors.
NUMERIC_FEATURES = [
    "core_size_nm",            # primary particle diameter
    "hydrodynamic_size_nm",    # size in medium (corona-inflated)
    "zeta_potential_mV",       # surface charge
    "surface_area_m2_g",       # specific surface area (BET)
    "band_gap_eV",             # electronic structure (oxidative potential)
    "cond_band_energy_eV",     # conduction band energy (metal-oxide QSAR)
    "dissolution_mg_L",        # ion-release proxy
    "concentration_mg_L",      # administered dose
    "exposure_time_h",         # assay duration
    "purity_pct",              # synthesis purity
]

#: Categorical descriptors.
CATEGORICAL_FEATURES = [
    "material_family",
    "coating",                 # surface functionalisation
    "cell_line",               # biological context
    "assay",                   # viability readout
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


@dataclass
class DatasetBundle:
    """A fully materialised dataset plus provenance metadata."""

    frame: pd.DataFrame
    numeric_features: list[str]
    categorical_features: list[str]
    target: str
    source: str  # "remote", "cache" or "synthetic"

    @property
    def X(self) -> pd.DataFrame:  # noqa: N802 (X is conventional in ML)
        return self.frame[self.numeric_features + self.categorical_features]

    @property
    def y(self) -> pd.Series:
        return self.frame[self.target]


# --------------------------------------------------------------------------- #
# Public entry point
# --------------------------------------------------------------------------- #

def load_dataset(config: DataConfig | None = None) -> DatasetBundle:
    """Return a :class:`DatasetBundle`, fetching or synthesising as needed."""

    config = config or DataConfig()
    frame: pd.DataFrame | None = None
    source = "synthetic"

    if config.remote_url:
        frame = _try_download(config.remote_url)
        if frame is not None:
            source = "remote"
            _cache(frame, config.cache_path)

    if frame is None and config.cache_path.exists():
        frame = pd.read_csv(config.cache_path)
        source = "cache"

    if frame is None:
        frame = make_synthetic_cohort(
            n=config.n_synthetic,
            viability_cutoff=config.viability_cutoff,
        )
        source = "synthetic"
        _cache(frame, config.cache_path)

    frame = _coerce_schema(frame, config.viability_cutoff)

    return DatasetBundle(
        frame=frame,
        numeric_features=[c for c in NUMERIC_FEATURES if c in frame.columns],
        categorical_features=[c for c in CATEGORICAL_FEATURES if c in frame.columns],
        target=TARGET_COLUMN,
        source=source,
    )


# --------------------------------------------------------------------------- #
# Remote / cache helpers
# --------------------------------------------------------------------------- #

def _try_download(url: str, timeout: int = 30) -> pd.DataFrame | None:
    """Best-effort CSV download. Returns ``None`` on any failure."""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:  # noqa: S310
            raw = resp.read()
        return pd.read_csv(io.BytesIO(raw))
    except (urllib.error.URLError, TimeoutError, ValueError, OSError):
        return None


def _cache(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def _coerce_schema(frame: pd.DataFrame, viability_cutoff: float) -> pd.DataFrame:
    """Normalise column names and derive the binary target if necessary."""
    frame = frame.copy()

    # Derive the binary target from a viability column if the dataset ships
    # continuous viability (%) rather than a pre-binarised label.
    if TARGET_COLUMN not in frame.columns:
        for cand in ("viability_pct", "cell_viability", "viability"):
            if cand in frame.columns:
                frame[TARGET_COLUMN] = (frame[cand] < viability_cutoff).astype(int)
                break

    if TARGET_COLUMN not in frame.columns:
        raise ValueError(
            f"Dataset has no '{TARGET_COLUMN}' column and no viability column "
            "to derive it from. Available columns: " + ", ".join(frame.columns)
        )

    frame[TARGET_COLUMN] = frame[TARGET_COLUMN].astype(int)
    return frame


# --------------------------------------------------------------------------- #
# Physicochemically-grounded synthetic generator
# --------------------------------------------------------------------------- #

# Per-family priors for the numeric descriptors: (mean, std, low, high).
# Values are chosen to sit inside ranges commonly reported in the
# nanotoxicology literature for each material class.
_FAMILY_PRIORS: dict[str, dict[str, tuple[float, float, float, float]]] = {
    "metal_oxide": {
        "core_size_nm": (30, 18, 5, 120),
        "zeta_potential_mV": (12, 20, -45, 55),
        "surface_area_m2_g": (90, 50, 10, 300),
        "band_gap_eV": (3.2, 0.6, 1.5, 5.5),
        "cond_band_energy_eV": (-4.3, 0.8, -6.5, -2.5),
        "dissolution_mg_L": (6.0, 5.0, 0.0, 40.0),
        "purity_pct": (98, 1.5, 90, 100),
    },
    "carbon": {
        "core_size_nm": (20, 12, 3, 80),
        "zeta_potential_mV": (-22, 15, -55, 20),
        "surface_area_m2_g": (210, 90, 40, 500),
        "band_gap_eV": (0.4, 0.5, 0.0, 2.0),
        "cond_band_energy_eV": (-4.8, 0.5, -6.0, -3.5),
        "dissolution_mg_L": (0.5, 0.8, 0.0, 5.0),
        "purity_pct": (95, 3.0, 80, 100),
    },
    "lipid": {
        "core_size_nm": (95, 35, 40, 220),
        "zeta_potential_mV": (-8, 18, -40, 40),
        "surface_area_m2_g": (25, 15, 5, 80),
        "band_gap_eV": (4.5, 0.4, 3.5, 5.5),
        "cond_band_energy_eV": (-5.5, 0.4, -6.5, -4.5),
        "dissolution_mg_L": (1.0, 1.2, 0.0, 8.0),
        "purity_pct": (97, 2.0, 88, 100),
    },
    "polymer": {
        "core_size_nm": (110, 45, 30, 300),
        "zeta_potential_mV": (5, 22, -50, 55),
        "surface_area_m2_g": (40, 25, 5, 130),
        "band_gap_eV": (3.8, 0.7, 2.0, 5.5),
        "cond_band_energy_eV": (-5.0, 0.6, -6.5, -3.5),
        "dissolution_mg_L": (0.8, 1.0, 0.0, 6.0),
        "purity_pct": (96, 2.5, 85, 100),
    },
}

_COATINGS = ("none", "PEG", "citrate", "carboxyl", "amine", "silica")
_CELL_LINES = ("A549", "HepG2", "HeLa", "RAW264.7", "BEAS-2B", "HEK293")
_ASSAYS = ("MTT", "MTS", "LDH", "WST-1", "CellTiterGlo")

# Relative biological sensitivity multipliers (toxic response scaling).
_CELL_SENSITIVITY = {
    "A549": 1.0, "HepG2": 1.15, "HeLa": 0.95,
    "RAW264.7": 1.3, "BEAS-2B": 1.1, "HEK293": 0.9,
}
# Protective effect of coatings (lower = more protective / more stealth).
_COATING_PROTECTION = {
    "none": 1.0, "PEG": 0.55, "citrate": 0.85,
    "carboxyl": 0.9, "amine": 1.25, "silica": 0.75,
}


def _truncated_normal(rng, mean, std, low, high, size):
    vals = rng.normal(mean, std, size)
    return np.clip(vals, low, high)


def _scalar_tn(rng, prior) -> float:
    """Draw a single truncated-normal scalar from a (mean, std, low, high) prior."""
    return float(_truncated_normal(rng, *prior, 1)[0])


def make_synthetic_cohort(n: int = 1500, viability_cutoff: float = 50.0,
                          random_state: int = 42) -> pd.DataFrame:
    """Generate a reproducible, mechanism-aware synthetic cohort.

    The latent viability is driven by a smooth, non-linear combination of
    descriptors reflecting three dominant cytotoxicity mechanisms reported in
    the literature:

    * **Ion dissolution** -- soluble-metal release (dose- and family-dependent).
    * **Oxidative stress** -- small size, high surface area and favourable
      conduction-band energy promote ROS generation.
    * **Membrane interaction** -- strongly cationic (positive zeta) particles
      disrupt membranes and are taken up aggressively.

    Coating, cell line, dose and exposure time modulate the response. Gaussian
    noise is added so the task is non-trivial and the problem stays realistic.
    """
    rng = np.random.default_rng(random_state)

    families = rng.choice(MATERIAL_FAMILIES, size=n,
                          p=[0.4, 0.2, 0.2, 0.2])
    rows: list[dict] = []

    for fam in families:
        priors = _FAMILY_PRIORS[fam]
        core = _scalar_tn(rng, priors["core_size_nm"])
        # Hydrodynamic size: corona inflates core size, more so for small/charged
        # particles; add medium-dependent aggregation.
        hydro = core * rng.uniform(1.1, 2.4) + rng.normal(0, 8)
        hydro = float(np.clip(hydro, core, 600))

        zeta = _scalar_tn(rng, priors["zeta_potential_mV"])
        sarea = _scalar_tn(rng, priors["surface_area_m2_g"])
        bgap = _scalar_tn(rng, priors["band_gap_eV"])
        cbe = _scalar_tn(rng, priors["cond_band_energy_eV"])
        diss = _scalar_tn(rng, priors["dissolution_mg_L"])
        purity = _scalar_tn(rng, priors["purity_pct"])

        conc = float(np.clip(rng.lognormal(mean=2.2, sigma=0.9), 0.5, 500))
        exposure = float(rng.choice([6, 12, 24, 48, 72]))
        coating = rng.choice(_COATINGS)
        cell = rng.choice(_CELL_LINES)
        assay = rng.choice(_ASSAYS)

        rows.append(dict(
            material_family=fam, core_size_nm=core, hydrodynamic_size_nm=hydro,
            zeta_potential_mV=zeta, surface_area_m2_g=sarea, band_gap_eV=bgap,
            cond_band_energy_eV=cbe, dissolution_mg_L=diss,
            concentration_mg_L=conc, exposure_time_h=exposure, purity_pct=purity,
            coating=coating, cell_line=cell, assay=assay,
        ))

    df = pd.DataFrame(rows)

    # --- Latent toxicity score (higher = more toxic) ----------------------- #
    # Oxidative-stress term: small size + large surface area + conduction band
    # energy overlapping the cellular redox window (~ -4.8 .. -4.2 eV).
    size_term = np.exp(-df["core_size_nm"] / 40.0)          # small -> ~1
    sa_term = df["surface_area_m2_g"] / 300.0
    redox_overlap = np.exp(-((df["cond_band_energy_eV"] + 4.5) ** 2) / 0.5)
    oxidative = 1.6 * size_term + 0.8 * sa_term + 1.2 * redox_overlap

    # Dissolution term: ion release scaled by dose (log) -- metal oxides matter
    # most here.
    dissolution = 0.9 * np.log1p(df["dissolution_mg_L"]) \
        * np.log1p(df["concentration_mg_L"]) / 4.0

    # Membrane term: cationic particles (positive zeta) are disruptive.
    membrane = 0.04 * np.clip(df["zeta_potential_mV"], 0, None)

    # Dose / time escalation.
    dose_time = 0.5 * np.log1p(df["concentration_mg_L"]) \
        + 0.012 * df["exposure_time_h"]

    score = oxidative + dissolution + membrane + dose_time

    # Biological + formulation modulation.
    score *= df["cell_line"].map(_CELL_SENSITIVITY).to_numpy()
    score *= df["coating"].map(_COATING_PROTECTION).to_numpy()
    # Impurities add a small toxic burden.
    score += 0.05 * (100.0 - df["purity_pct"])

    # Irreducible biological/assay noise.
    score += rng.normal(0, 0.9, size=len(df))

    # Map latent score -> viability (%) via a logistic response, then invert.
    # Centre/scale chosen so the cohort is reasonably balanced.
    p_toxic = 1.0 / (1.0 + np.exp(-(score - score.mean()) / (score.std() + 1e-9)))
    viability = 100.0 * (1.0 - p_toxic) + rng.normal(0, 5, size=len(df))
    df["viability_pct"] = np.clip(viability, 0, 100).round(2)
    df[TARGET_COLUMN] = (df["viability_pct"] < viability_cutoff).astype(int)

    return df


In [ ]:
%%writefile nanotox/features/__init__.py
"""Feature engineering for nanotox."""

from __future__ import annotations

from .engineering import (
    DerivedFeatures,
    build_preprocessor,
    engineered_feature_names,
)

__all__ = ["DerivedFeatures", "build_preprocessor", "engineered_feature_names"]


In [ ]:
%%writefile nanotox/features/engineering.py
"""Physicochemistry-aware feature engineering.

Two layers:

1. :class:`DerivedFeatures` -- a stateless ``sklearn`` transformer that adds
   domain-motivated derived descriptors (ratios, log-doses, interaction terms)
   that encode nano-QSAR priors the raw columns only imply.
2. :func:`build_preprocessor` -- assembles the full
   ``ColumnTransformer`` (impute + scale numerics, one-hot categoricals) that
   every model in the zoo shares, so comparisons are apples-to-apples.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Names of the engineered columns added by DerivedFeatures (in order).
_DERIVED_COLUMNS = [
    "log_concentration",
    "log_dissolution",
    "corona_inflation_ratio",   # hydrodynamic / core size
    "specific_reactivity",      # surface_area / core_size
    "abs_zeta",                 # |zeta| -- colloidal (in)stability
    "cationic_flag",            # zeta > +10 mV
    "dose_x_dissolution",       # ion-release burden at dose
    "size_x_surface",           # size-normalised surface reactivity
    "redox_window_proximity",   # closeness of cond. band to redox window
]


class DerivedFeatures(BaseEstimator, TransformerMixin):
    """Append domain-motivated derived descriptors to the numeric block.

    The transformer is pure (no fitted state beyond column bookkeeping) so it
    is safe to place upstream of imputation inside a CV pipeline without
    leakage. Missing inputs propagate as NaN and are handled downstream by the
    imputer.
    """

    def __init__(self) -> None:
        self.input_columns_: list[str] = []
        self.output_columns_: list[str] = []

    def fit(self, X: pd.DataFrame, y=None):  # noqa: N803
        self.input_columns_ = list(X.columns)
        self.output_columns_ = list(X.columns) + _DERIVED_COLUMNS
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:  # noqa: N803
        X = X.copy()

        def col(name: str) -> pd.Series:
            return X[name] if name in X.columns else pd.Series(np.nan, index=X.index)

        conc = col("concentration_mg_L")
        diss = col("dissolution_mg_L")
        core = col("core_size_nm")
        hydro = col("hydrodynamic_size_nm")
        sarea = col("surface_area_m2_g")
        zeta = col("zeta_potential_mV")
        cbe = col("cond_band_energy_eV")

        eps = 1e-6
        X["log_concentration"] = np.log1p(conc.clip(lower=0))
        X["log_dissolution"] = np.log1p(diss.clip(lower=0))
        X["corona_inflation_ratio"] = hydro / (core + eps)
        X["specific_reactivity"] = sarea / (core + eps)
        X["abs_zeta"] = zeta.abs()
        X["cationic_flag"] = (zeta > 10).astype(float)
        X["dose_x_dissolution"] = np.log1p(conc.clip(lower=0)) * np.log1p(diss.clip(lower=0))
        X["size_x_surface"] = sarea / (core**2 + eps)
        # Cellular redox window centred near -4.5 eV; closeness -> ROS potential.
        X["redox_window_proximity"] = np.exp(-((cbe + 4.5) ** 2) / 0.5)

        return X

    def get_feature_names_out(self, input_features=None):  # noqa: D401
        return np.asarray(self.output_columns_, dtype=object)


def engineered_feature_names(numeric_features: list[str]) -> list[str]:
    """The numeric feature names after :class:`DerivedFeatures` expansion."""
    return list(numeric_features) + _DERIVED_COLUMNS


def build_preprocessor(numeric_features: list[str],
                       categorical_features: list[str]) -> Pipeline:
    """Full preprocessing pipeline shared by every model.

    numeric  : DerivedFeatures -> median impute -> standardise
    category : most-frequent impute -> one-hot (dense, ignore unknown)
    """
    numeric_out = engineered_feature_names(numeric_features)

    numeric_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    column_transform = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_out),
            ("cat", categorical_pipe, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )

    # DerivedFeatures runs first so the engineered numeric columns exist before
    # the ColumnTransformer selects them by name.
    return Pipeline(steps=[
        ("derive", DerivedFeatures()),
        ("columns", column_transform),
    ])


In [ ]:
%%writefile nanotox/models/__init__.py
"""Model zoo for nanotox."""

from __future__ import annotations

from .zoo import ModelSpec, available_models, build_model_specs

__all__ = ["ModelSpec", "available_models", "build_model_specs"]


In [ ]:
%%writefile nanotox/models/zoo.py
"""The model zoo and their hyper-parameter search spaces.

Each entry is a :class:`ModelSpec` bundling an estimator factory with a
``RandomizedSearchCV`` parameter distribution. Optional gradient-boosting
libraries (XGBoost, LightGBM) are included only if importable, so the pipeline
runs on a minimal install and gets stronger when they are present.

Parameter-grid keys are prefixed with ``clf__`` because every estimator is the
final ``clf`` step of a shared preprocessing ``Pipeline`` (see
:mod:`nanotox.pipeline`).
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable

from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from ..config import RANDOM_STATE

# Optional boosters -- imported defensively.
try:  # pragma: no cover - availability depends on environment
    from xgboost import XGBClassifier
    _HAS_XGB = True
except Exception:  # noqa: BLE001
    _HAS_XGB = False

try:  # pragma: no cover
    from lightgbm import LGBMClassifier
    _HAS_LGBM = True
except Exception:  # noqa: BLE001
    _HAS_LGBM = False


@dataclass
class ModelSpec:
    """An estimator plus its hyper-parameter search distribution."""

    name: str
    factory: Callable[[], BaseEstimator]
    param_distributions: dict[str, Any] = field(default_factory=dict)
    # Whether this estimator needs scaled/dense input (all do here, since the
    # shared preprocessor always scales + densifies).
    notes: str = ""


def build_model_specs(random_state: int = RANDOM_STATE) -> dict[str, ModelSpec]:
    """Construct every available model spec, keyed by short name."""
    specs: dict[str, ModelSpec] = {}

    specs["logistic"] = ModelSpec(
        name="LogisticRegression",
        factory=lambda: LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=random_state),
        param_distributions={
            "clf__C": loguniform(1e-3, 1e2),
        },
        notes="Linear baseline (L2); interpretable reference point.",
    )

    specs["svm_rbf"] = ModelSpec(
        name="SVM (RBF)",
        factory=lambda: SVC(
            kernel="rbf", probability=True, class_weight="balanced",
            random_state=random_state),
        param_distributions={
            "clf__C": loguniform(1e-1, 1e3),
            "clf__gamma": loguniform(1e-4, 1e0),
        },
        notes="Kernel method; strong on smooth non-linear boundaries.",
    )

    specs["knn"] = ModelSpec(
        name="k-NN",
        factory=lambda: KNeighborsClassifier(),
        param_distributions={
            "clf__n_neighbors": randint(3, 35),
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2],
        },
        notes="Instance-based baseline.",
    )

    specs["random_forest"] = ModelSpec(
        name="RandomForest",
        factory=lambda: RandomForestClassifier(
            class_weight="balanced_subsample", random_state=random_state,
            n_jobs=1),
        param_distributions={
            "clf__n_estimators": randint(200, 800),
            "clf__max_depth": randint(3, 25),
            "clf__min_samples_leaf": randint(1, 12),
            "clf__max_features": uniform(0.3, 0.7),
        },
        notes="Robust bagged-tree workhorse of nano-QSAR.",
    )

    specs["extra_trees"] = ModelSpec(
        name="ExtraTrees",
        factory=lambda: ExtraTreesClassifier(
            class_weight="balanced_subsample", random_state=random_state,
            n_jobs=1),
        param_distributions={
            "clf__n_estimators": randint(200, 800),
            "clf__max_depth": randint(3, 25),
            "clf__min_samples_leaf": randint(1, 12),
            "clf__max_features": uniform(0.3, 0.7),
        },
        notes="Extremely randomised trees; lower variance.",
    )

    specs["grad_boost"] = ModelSpec(
        name="GradientBoosting",
        factory=lambda: GradientBoostingClassifier(random_state=random_state),
        param_distributions={
            "clf__n_estimators": randint(150, 600),
            "clf__learning_rate": loguniform(1e-2, 3e-1),
            "clf__max_depth": randint(2, 6),
            "clf__subsample": uniform(0.6, 0.4),
        },
        notes="sklearn gradient boosting; always available.",
    )

    if _HAS_XGB:
        specs["xgboost"] = ModelSpec(
            name="XGBoost",
            factory=lambda: XGBClassifier(
                objective="binary:logistic", eval_metric="logloss",
                tree_method="hist", random_state=random_state, n_jobs=1),
            param_distributions={
                "clf__n_estimators": randint(200, 800),
                "clf__learning_rate": loguniform(1e-2, 3e-1),
                "clf__max_depth": randint(2, 8),
                "clf__subsample": uniform(0.6, 0.4),
                "clf__colsample_bytree": uniform(0.6, 0.4),
                "clf__reg_lambda": loguniform(1e-2, 1e1),
            },
            notes="Gradient-boosted trees; typically top performer.",
        )

    if _HAS_LGBM:
        specs["lightgbm"] = ModelSpec(
            name="LightGBM",
            factory=lambda: LGBMClassifier(
                objective="binary", random_state=random_state, n_jobs=1,
                verbose=-1),
            param_distributions={
                "clf__n_estimators": randint(200, 800),
                "clf__learning_rate": loguniform(1e-2, 3e-1),
                "clf__num_leaves": randint(15, 120),
                "clf__max_depth": randint(-1, 12),
                "clf__subsample": uniform(0.6, 0.4),
                "clf__colsample_bytree": uniform(0.6, 0.4),
                "clf__reg_lambda": loguniform(1e-2, 1e1),
            },
            notes="Histogram gradient boosting; fast and accurate.",
        )

    return specs


def available_models(random_state: int = RANDOM_STATE) -> list[str]:
    """Names of models usable in the current environment."""
    return list(build_model_specs(random_state).keys())


In [ ]:
%%writefile nanotox/evaluate.py
"""Metrics, comparison tables and publication-ready plots."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


def classification_metrics(y_true, y_pred, y_score) -> dict[str, float]:
    """Compute the standard panel of binary-classification metrics.

    MCC and balanced accuracy are included because nanotoxicity datasets are
    frequently class-imbalanced, where raw accuracy is misleading.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }
    # Probabilistic metrics need both classes present in y_true.
    if y_score is not None and len(np.unique(y_true)) > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        metrics["pr_auc"] = average_precision_score(y_true, y_score)
    else:
        metrics["roc_auc"] = float("nan")
        metrics["pr_auc"] = float("nan")
    return metrics


def leaderboard(results: dict[str, dict]) -> pd.DataFrame:
    """Build a sorted comparison table from per-model result dicts.

    Each value in ``results`` is expected to carry a ``cv_mean`` / ``cv_std``
    mapping and a ``holdout`` metric mapping.
    """
    rows = []
    for name, res in results.items():
        row = {"model": name}
        for k, v in res.get("cv_mean", {}).items():
            row[f"cv_{k}"] = v
        for k, v in res.get("cv_std", {}).items():
            row[f"cv_{k}_std"] = v
        for k, v in res.get("holdout", {}).items():
            row[f"holdout_{k}"] = v
        rows.append(row)
    df = pd.DataFrame(rows)
    sort_key = "holdout_roc_auc" if "holdout_roc_auc" in df.columns else "cv_roc_auc"
    if sort_key in df.columns:
        df = df.sort_values(sort_key, ascending=False, na_position="last")
    return df.reset_index(drop=True)


# --------------------------------------------------------------------------- #
# Plots (matplotlib, Agg-safe)
# --------------------------------------------------------------------------- #

def _ensure_mpl():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    return plt


def plot_roc_curves(roc_data: dict[str, tuple], out_path: Path) -> None:
    """ROC overlay for every model. ``roc_data[name] = (y_true, y_score)``."""
    plt = _ensure_mpl()
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, (y_true, y_score) in roc_data.items():
        if y_score is None or len(np.unique(y_true)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves (held-out test set)")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_model_comparison(board: pd.DataFrame, out_path: Path,
                          metric: str = "holdout_roc_auc") -> None:
    """Bar chart of a chosen metric across models."""
    if metric not in board.columns:
        return
    plt = _ensure_mpl()
    data = board.dropna(subset=[metric]).sort_values(metric)
    fig, ax = plt.subplots(figsize=(7, 0.5 * len(data) + 2))
    ax.barh(data["model"], data[metric], color="#3b7dd8")
    ax.set_xlabel(metric)
    ax.set_xlim(0, 1)
    ax.set_title(f"Model comparison by {metric}")
    for y, v in enumerate(data[metric]):
        ax.text(v + 0.01, y, f"{v:.3f}", va="center", fontsize=8)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_confusion(y_true, y_pred, out_path: Path, title: str) -> None:
    """Confusion matrix heat-map for the best model."""
    plt = _ensure_mpl()
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["non-toxic", "cytotoxic"])
    ax.set_yticks([0, 1], ["non-toxic", "cytotoxic"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


In [ ]:
%%writefile nanotox/pipeline.py
"""End-to-end training, model selection and benchmarking.

Core design choices (all publication-relevant):

* **Nested cross-validation** for unbiased model comparison: an inner
  ``RandomizedSearchCV`` tunes hyper-parameters, an outer ``StratifiedKFold``
  estimates generalisation. This avoids the optimistic bias of tuning and
  evaluating on the same folds.
* **A single untouched hold-out set** gives a final, honest point estimate and
  the data for ROC/confusion plots.
* **Leave-one-material-family-out** directly measures the cross-family
  generalisation gap highlighted in the research brief.
* Everything is seeded and the resolved config is serialised with the results.
"""

from __future__ import annotations

import json
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline

from .config import PipelineConfig
from .data import DatasetBundle, load_dataset
from .evaluate import (
    classification_metrics,
    leaderboard,
    plot_confusion,
    plot_model_comparison,
    plot_roc_curves,
)
from .features import build_preprocessor
from .models import build_model_specs

_SCORING = ["roc_auc", "average_precision", "balanced_accuracy", "f1", "matthews_corrcoef"]


@dataclass
class BenchmarkResult:
    """Everything a run produces, ready to serialise."""

    leaderboard: pd.DataFrame
    per_model: dict[str, dict]
    best_model_name: str
    best_estimator: Pipeline
    dataset_source: str
    config: dict

    def save(self, results_dir: Path) -> None:
        results_dir.mkdir(parents=True, exist_ok=True)
        self.leaderboard.to_csv(results_dir / "leaderboard.csv", index=False)
        summary = {
            "best_model": self.best_model_name,
            "dataset_source": self.dataset_source,
            "config": self.config,
            "per_model": {
                k: {kk: vv for kk, vv in v.items() if kk != "roc_data"}
                for k, v in self.per_model.items()
            },
        }
        (results_dir / "summary.json").write_text(json.dumps(summary, indent=2, default=float))


def _make_estimator(preprocessor: Pipeline, clf) -> Pipeline:
    """Glue the shared preprocessor to a classifier as the ``clf`` step."""
    return Pipeline(steps=[("pre", preprocessor), ("clf", clf)])


def run_benchmark(config: PipelineConfig | None = None,
                  bundle: DatasetBundle | None = None,
                  make_plots: bool = True) -> BenchmarkResult:
    """Run the full benchmark and return a :class:`BenchmarkResult`."""
    config = config or PipelineConfig()
    bundle = bundle or load_dataset(config.data)

    X, y = bundle.X, bundle.y
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.data.test_size, stratify=y,
        random_state=config.random_state,
    )

    specs = build_model_specs(config.random_state)
    if config.models:
        specs = {k: v for k, v in specs.items() if k in config.models}
        if not specs:
            raise ValueError(f"No requested models available: {config.models}")

    outer_cv = StratifiedKFold(
        n_splits=config.cv.outer_folds, shuffle=True,
        random_state=config.random_state)
    inner_cv = StratifiedKFold(
        n_splits=config.cv.inner_folds, shuffle=True,
        random_state=config.random_state)

    per_model: dict[str, dict] = {}
    roc_data: dict[str, tuple] = {}

    for key, spec in specs.items():
        t0 = time.time()
        preprocessor = build_preprocessor(
            bundle.numeric_features, bundle.categorical_features)
        estimator = _make_estimator(preprocessor, spec.factory())

        # Inner search wrapped as the estimator for the outer loop => nested CV.
        search = RandomizedSearchCV(
            estimator,
            param_distributions=spec.param_distributions,
            n_iter=config.cv.n_iter_search,
            scoring=config.cv.scoring,
            cv=inner_cv,
            random_state=config.random_state,
            n_jobs=-1,
            refit=True,
            error_score="raise",
        )

        # Unbiased generalisation estimate via nested CV on the training split.
        # IMPORTANT: only ONE level of the nested loop is parallelised. The
        # inner RandomizedSearchCV already uses n_jobs=-1, so the outer
        # cross_validate runs serially (n_jobs=1). Parallelising both, while
        # the tree/boosting estimators are *also* multi-threaded, oversubscribes
        # the CPU badly (dozens of threads per core) and is dramatically slower.
        nested = cross_validate(
            search, X_train, y_train, cv=outer_cv, scoring=_SCORING,
            n_jobs=1, return_estimator=False,
        )
        cv_mean = {_clean(k): float(np.mean(v)) for k, v in nested.items()
                   if k.startswith("test_")}
        cv_std = {_clean(k): float(np.std(v)) for k, v in nested.items()
                  if k.startswith("test_")}

        # Refit the tuned pipeline on all training data, score the hold-out.
        search.fit(X_train, y_train)
        best = search.best_estimator_
        y_pred = best.predict(X_test)
        y_score = _safe_scores(best, X_test)
        holdout = classification_metrics(y_test, y_pred, y_score)

        per_model[spec.name] = {
            "key": key,
            "cv_mean": cv_mean,
            "cv_std": cv_std,
            "holdout": holdout,
            "best_params": search.best_params_,
            "fit_seconds": round(time.time() - t0, 2),
            "roc_data": (y_test.to_numpy(), y_score),
            "_estimator": best,
        }
        roc_data[spec.name] = (y_test.to_numpy(), y_score)

    board = leaderboard(per_model)
    best_name = board.iloc[0]["model"]
    best_estimator = per_model[best_name]["_estimator"]

    # Strip the heavy estimator object out of the serialisable dict.
    serialisable = {k: {kk: vv for kk, vv in v.items() if kk != "_estimator"}
                    for k, v in per_model.items()}

    if make_plots:
        rdir = config.results_dir
        plot_roc_curves(roc_data, rdir / "roc_curves.png")
        plot_model_comparison(board, rdir / "model_comparison.png")
        y_pred_best = best_estimator.predict(X_test)
        plot_confusion(y_test, y_pred_best, rdir / "confusion_best.png",
                       title=f"Confusion matrix -- {best_name}")

    return BenchmarkResult(
        leaderboard=board,
        per_model=serialisable,
        best_model_name=best_name,
        best_estimator=best_estimator,
        dataset_source=bundle.source,
        config=config.to_dict(),
    )


def leave_one_family_out(config: PipelineConfig | None = None,
                         bundle: DatasetBundle | None = None,
                         model_key: str = "random_forest") -> pd.DataFrame:
    """Measure cross-family generalisation.

    Train on all-but-one material family, test on the held-out family. A large
    drop versus the pooled hold-out AUC quantifies the "generalisation
    inability across diverse material families" gap.
    """
    config = config or PipelineConfig()
    bundle = bundle or load_dataset(config.data)
    frame = bundle.frame
    if "material_family" not in frame.columns:
        raise ValueError("Dataset lacks 'material_family'; cannot run LOFO.")

    specs = build_model_specs(config.random_state)
    if model_key not in specs:
        model_key = "grad_boost"  # always-available fallback
    spec = specs[model_key]

    rows = []
    for fam in sorted(frame["material_family"].unique()):
        train = frame[frame["material_family"] != fam]
        test = frame[frame["material_family"] == fam]
        if test[bundle.target].nunique() < 2 or len(test) < 10:
            continue
        feats = bundle.numeric_features + bundle.categorical_features
        pre = build_preprocessor(bundle.numeric_features, bundle.categorical_features)
        est = _make_estimator(pre, spec.factory())
        est.fit(train[feats], train[bundle.target])
        y_score = _safe_scores(est, test[feats])
        y_pred = est.predict(test[feats])
        m = classification_metrics(test[bundle.target].to_numpy(), y_pred, y_score)
        rows.append({"held_out_family": fam, "n_test": len(test),
                     "roc_auc": m["roc_auc"], "balanced_accuracy": m["balanced_accuracy"],
                     "mcc": m["mcc"]})
    return pd.DataFrame(rows)


def feature_importance(result: BenchmarkResult,
                       bundle: DatasetBundle) -> pd.DataFrame | None:
    """Permutation importance for the best estimator (model-agnostic).

    Permutation importance is preferred over impurity-based importance because
    it is unbiased w.r.t. feature cardinality and works for any estimator.
    """
    from sklearn.inspection import permutation_importance

    est = result.best_estimator
    X, y = bundle.X, bundle.y
    try:
        r = permutation_importance(
            est, X, y, n_repeats=10, random_state=42, n_jobs=-1,
            scoring="roc_auc")
    except Exception:  # noqa: BLE001
        return None
    df = pd.DataFrame({
        "feature": X.columns,
        "importance_mean": r.importances_mean,
        "importance_std": r.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    return df


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #

def _clean(metric_key: str) -> str:
    return metric_key.replace("test_", "")


def _safe_scores(estimator, X) -> np.ndarray | None:
    """Positive-class scores via predict_proba or decision_function."""
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X)[:, 1]
    if hasattr(estimator, "decision_function"):
        return estimator.decision_function(X)
    return None


In [ ]:
%%writefile nanotox/cli.py
"""Command-line interface: ``nanotox ...``.

Subcommands
-----------
benchmark   Run the full nested-CV benchmark across the model zoo.
lofo        Leave-one-material-family-out generalisation test.
info        Print dataset provenance and available models.
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

from .config import DataConfig, CVConfig, PipelineConfig
from .data import load_dataset
from .models import available_models
from .pipeline import feature_importance, leave_one_family_out, run_benchmark


# Fast preset: quick folds/search and a subset of inexpensive models, for
# iteration on modest hardware. Heavy models (SVM, GradientBoosting) and the
# full search are reserved for the publication-grade default run.
_FAST_MODELS = ["logistic", "random_forest", "xgboost", "lightgbm"]


def _build_config(args) -> PipelineConfig:
    data = DataConfig()
    if args.remote_url:
        data.remote_url = args.remote_url
    if args.n_synthetic:
        data.n_synthetic = args.n_synthetic
    if args.viability is not None:
        data.viability_cutoff = args.viability

    fast = getattr(args, "fast", False)
    # --fast only fills values the user did not set explicitly, so explicit
    # flags always win.
    outer = args.outer_folds if args.outer_folds is not None else (3 if fast else 5)
    inner = args.inner_folds if args.inner_folds is not None else (2 if fast else 3)
    n_iter = args.n_iter if args.n_iter is not None else (8 if fast else 40)
    cv = CVConfig(outer_folds=outer, inner_folds=inner, n_iter_search=n_iter)

    if args.models:
        models = args.models.split(",")
    elif fast:
        models = list(_FAST_MODELS)
    else:
        models = None

    return PipelineConfig(
        data=data, cv=cv,
        results_dir=Path(args.results_dir),
        models=models,
    )


def _cmd_benchmark(args) -> int:
    config = _build_config(args)
    result = run_benchmark(config, make_plots=not args.no_plots)
    result.save(config.results_dir)

    print(f"\nDataset source: {result.dataset_source}")
    print(f"Best model    : {result.best_model_name}\n")
    cols = [c for c in ["model", "cv_roc_auc", "cv_roc_auc_std",
                        "holdout_roc_auc", "holdout_pr_auc", "holdout_mcc",
                        "holdout_balanced_accuracy"]
            if c in result.leaderboard.columns]
    with_fmt = result.leaderboard[cols].copy()
    for c in cols[1:]:
        with_fmt[c] = with_fmt[c].map(lambda v: f"{v:.3f}")
    print(with_fmt.to_string(index=False))

    if args.importance:
        bundle = load_dataset(config.data)
        imp = feature_importance(result, bundle)
        if imp is not None:
            print("\nTop permutation-importance features:")
            print(imp.head(12).to_string(index=False))
            imp.to_csv(config.results_dir / "feature_importance.csv", index=False)

    print(f"\nArtifacts written to: {config.results_dir}")
    return 0


def _cmd_lofo(args) -> int:
    config = _build_config(args)
    table = leave_one_family_out(config, model_key=args.model_key)
    print("\nLeave-one-material-family-out generalisation:\n")
    fmt = table.copy()
    for c in ("roc_auc", "balanced_accuracy", "mcc"):
        if c in fmt.columns:
            fmt[c] = fmt[c].map(lambda v: f"{v:.3f}")
    print(fmt.to_string(index=False))
    config.results_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(config.results_dir / "lofo.csv", index=False)
    return 0


def _cmd_info(args) -> int:
    config = _build_config(args)
    bundle = load_dataset(config.data)
    print(f"Dataset source      : {bundle.source}")
    print(f"Rows                : {len(bundle.frame)}")
    print(f"Positive rate       : {bundle.y.mean():.3f}")
    print(f"Numeric features    : {', '.join(bundle.numeric_features)}")
    print(f"Categorical features: {', '.join(bundle.categorical_features)}")
    print(f"Available models    : {', '.join(available_models())}")
    return 0


def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="nanotox",
                                description="Nanoparticle cytotoxicity ML benchmark")
    # Shared options.
    common = argparse.ArgumentParser(add_help=False)
    common.add_argument("--remote-url", default=None,
                        help="CSV URL to import (falls back to synthetic if unreachable)")
    common.add_argument("--n-synthetic", type=int, default=0,
                        help="Synthetic cohort size when generating data")
    common.add_argument("--viability", type=float, default=None,
                        help="Viability%% cutoff for deriving the binary target")
    common.add_argument("--outer-folds", type=int, default=None,
                        help="Outer CV folds (default 5, or 3 with --fast)")
    common.add_argument("--inner-folds", type=int, default=None,
                        help="Inner CV folds (default 3, or 2 with --fast)")
    common.add_argument("--n-iter", type=int, default=None,
                        help="RandomizedSearchCV iterations per model "
                             "(default 40, or 8 with --fast)")
    common.add_argument("--fast", action="store_true",
                        help="Quick preset: fewer folds/iterations and a subset "
                             "of inexpensive models (skips SVM/GradientBoosting)")
    common.add_argument("--models", default=None,
                        help="Comma-separated subset of model keys")
    common.add_argument("--results-dir", default="results")

    sub = p.add_subparsers(dest="command", required=True)

    b = sub.add_parser("benchmark", parents=[common], help="Full benchmark")
    b.add_argument("--no-plots", action="store_true")
    b.add_argument("--importance", action="store_true",
                   help="Compute permutation importance for the best model")
    b.set_defaults(func=_cmd_benchmark)

    lo = sub.add_parser("lofo", parents=[common],
                        help="Leave-one-material-family-out test")
    lo.add_argument("--model-key", default="random_forest")
    lo.set_defaults(func=_cmd_lofo)

    i = sub.add_parser("info", parents=[common], help="Dataset / model info")
    i.set_defaults(func=_cmd_info)

    return p


def main(argv: list[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv if argv is not None else sys.argv[1:])
    return args.func(args)


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, os.getcwd())   # so `import nanotox` resolves to the files above
import importlib
import nanotox; importlib.reload(nanotox)
from nanotox.config import DataConfig, CVConfig, PipelineConfig
from nanotox.data import load_dataset, make_synthetic_cohort
from nanotox.data.loader import NUMERIC_FEATURES, CATEGORICAL_FEATURES, DatasetBundle, _coerce_schema
from nanotox.features import build_preprocessor
from nanotox.models import available_models
from nanotox.pipeline import run_benchmark, leave_one_family_out, feature_importance
print("nanotox ready | models:", available_models())


## 2. Inline-plot helper

`run_benchmark` writes PNGs to `results/`; this shows them in the notebook.

In [ ]:
from IPython.display import Image, display
import pandas as pd, os

def show_results(results_dir="results"):
    for name in ["model_comparison.png", "roc_curves.png", "confusion_best.png"]:
        p = os.path.join(results_dir, name)
        if os.path.exists(p):
            print(name); display(Image(filename=p))
    lb = os.path.join(results_dir, "leaderboard.csv")
    if os.path.exists(lb):
        display(pd.read_csv(lb))


## 3. Real data — dataset registry & import helpers

The pipeline trains on any table with the schema below. Missing columns are
tolerated (median/most-frequent imputed). The target `cytotoxic` (0/1) is
derived from a `viability_pct` column if not already present.

**Expected schema**

| Kind | Columns |
|---|---|
| Numeric | `core_size_nm`, `hydrodynamic_size_nm`, `zeta_potential_mV`, `surface_area_m2_g`, `band_gap_eV`, `cond_band_energy_eV`, `dissolution_mg_L`, `concentration_mg_L`, `exposure_time_h`, `purity_pct` |
| Categorical | `material_family`, `coating`, `cell_line`, `assay` |
| Target | `cytotoxic` (0/1), or `viability_pct` (%) to derive it |

**Where each named dataset fits** (and how to pull it)

| Source | What it gives | Access route |
|---|---|---|
| **eNanoMapper** | Nanomaterial physchem + in-vitro tox (FAIR) | REST API `https://search.data.enanomapper.net/` / Ambit; export CSV → `map_columns` |
| **NanoInformatics (NIkC) / S2NANO / caNanoLab** | Curated physchem + cytotoxicity across families | Portal CSV export (often manual download) → `map_columns` |
| **PubChem** | Compound/substance physchem for shells & ligands | REST PUG (`fetch_pubchem`) |
| **Materials Project** | Band gap, conduction-band energy, lattice/crystal | API **key required**: `mp-api` client (`fetch_materials_project`) |
| **ChEMBL** | Bioactivity of shell chemistries / ligands | `chembl_webresource_client` (`fetch_chembl`) |
| **PRIDE / MassIVE** | Protein-corona proteomics (raw MS) | Large raw files — summarise to per-NP features offline, then join |
| **Open TG-GATEs / LINCS L1000** | Transcriptomic response to perturbations | Bulk omics — derive per-sample features offline, then join as extra columns |
| **Your own / a new CSV** | Anything | `load_url`, `upload_csv` |

The omics repositories (PRIDE, MassIVE, TG-GATEs, LINCS) are **not** direct
tabular downloads: you reduce them to per-nanoparticle features offline and
join them onto the physchem table as extra numeric columns — the pipeline then
uses them automatically.


In [ ]:
import io, json, urllib.request, urllib.error
import pandas as pd

# ---- Generic importers -----------------------------------------------------
def load_url(url: str) -> pd.DataFrame:
    """Download a CSV/TSV from any public URL."""
    sep = "\t" if url.lower().endswith((".tsv", ".tab")) else ","
    return pd.read_csv(url, sep=sep)

def upload_csv() -> pd.DataFrame:
    """Prompt a Colab file upload and read the first CSV/TSV chosen."""
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    sep = "\t" if name.lower().endswith((".tsv", ".tab")) else ","
    return pd.read_csv(io.BytesIO(up[name]), sep=sep)

# ---- Column mapping to the pipeline schema ---------------------------------
def map_columns(df: pd.DataFrame, mapping: dict, viability_cutoff: float = 50.0) -> pd.DataFrame:
    """Rename a real dataset's columns to the nanotox schema.

    `mapping` is {your_column: schema_column}. Unmapped columns are kept (and
    ignored by the model unless they match a schema name). The binary target is
    derived from `viability_pct` if `cytotoxic` is absent.
    """
    out = df.rename(columns=mapping).copy()
    return _coerce_schema(out, viability_cutoff)

def bundle_from_frame(df: pd.DataFrame, viability_cutoff: float = 50.0) -> DatasetBundle:
    """Wrap a schema-mapped DataFrame as a DatasetBundle the pipeline accepts."""
    frame = _coerce_schema(df, viability_cutoff)
    return DatasetBundle(
        frame=frame,
        numeric_features=[c for c in NUMERIC_FEATURES if c in frame.columns],
        categorical_features=[c for c in CATEGORICAL_FEATURES if c in frame.columns],
        target="cytotoxic",
        source="real",
    )
print("importers ready: load_url, upload_csv, map_columns, bundle_from_frame")


In [ ]:
# ---- Source-specific fetchers (best-effort; each degrades gracefully) ------
def fetch_pubchem(cids, props=("MolecularWeight", "XLogP", "TPSA")):
    """Pull physchem properties for a list of PubChem CIDs via the REST PUG API."""
    import pandas as pd, urllib.request, io
    cid_str = ",".join(str(c) for c in cids)
    prop_str = ",".join(props)
    url = (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}"
           f"/property/{prop_str}/CSV")
    return pd.read_csv(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))

def fetch_chembl(target_or_molecule: str, limit: int = 200):
    """Pull bioactivities from ChEMBL (needs `chembl_webresource_client`)."""
    try:
        from chembl_webresource_client.new_client import new_client
    except ImportError:
        !pip -q install chembl_webresource_client
        from chembl_webresource_client.new_client import new_client
    import pandas as pd
    acts = new_client.activity.filter(
        molecule_chembl_id=target_or_molecule).only(
        ["standard_type", "standard_value", "standard_units"])[:limit]
    return pd.DataFrame(acts)

def fetch_materials_project(formulas, api_key=None):
    """Band gap & conduction-band proxies from the Materials Project (API KEY REQUIRED).

    Get a free key at https://materialsproject.org/api then pass api_key=...
    """
    if not api_key:
        raise ValueError("Materials Project needs an API key: https://materialsproject.org/api")
    from mp_api.client import MPRester
    import pandas as pd
    rows = []
    with MPRester(api_key) as mpr:
        for f in formulas:
            docs = mpr.summary.search(formula=f, fields=["formula_pretty", "band_gap"])
            for d in docs:
                rows.append({"formula": d.formula_pretty, "band_gap_eV": d.band_gap})
    return pd.DataFrame(rows)

def fetch_enanomapper(query_url="https://search.data.enanomapper.net/"):
    """Placeholder: eNanoMapper exposes an Ambit REST API. Export a study as CSV
    from the portal (or query the API) and load it with load_url / upload_csv,
    then map_columns to the schema."""
    print("eNanoMapper: export a CSV from", query_url, "then use load_url/upload_csv + map_columns")
print("fetchers ready: fetch_pubchem, fetch_chembl, fetch_materials_project, fetch_enanomapper")


### 3a. Choose your data source

Pick ONE cell to define `bundle`. Default is synthetic so the notebook runs
immediately. Switch to a real source for your paper.

In [ ]:
# --- Option A (default): synthetic, mechanism-grounded cohort ---------------
bundle = load_dataset(DataConfig(n_synthetic=1500, cache_path=Path("nano_synth_cache.csv")))
print("source:", bundle.source, "| rows:", len(bundle.frame),
      "| positive rate:", round(bundle.y.mean(), 3))
bundle.frame.head()


In [ ]:
# --- Option B: your own CSV from a URL --------------------------------------
# raw = load_url("https://raw.githubusercontent.com/<you>/<repo>/main/nano.csv")
# mapping = {  # your_column: schema_column
#     "size_nm": "core_size_nm", "zeta_mV": "zeta_potential_mV",
#     "conc_ugml": "concentration_mg_L", "viability": "viability_pct",
#     "material": "material_family", "cell": "cell_line",
# }
# bundle = bundle_from_frame(map_columns(raw, mapping))
# print(bundle.source, len(bundle.frame)); bundle.frame.head()


In [ ]:
# --- Option C: upload a CSV from your computer ------------------------------
# raw = upload_csv()
# mapping = { ... }                       # map your columns -> schema (see 3)
# bundle = bundle_from_frame(map_columns(raw, mapping))
# bundle.frame.head()


## 4. Dataset / model info

In [ ]:
print("Dataset source      :", bundle.source)
print("Rows                :", len(bundle.frame))
print("Positive (toxic) rate:", round(bundle.y.mean(), 3))
print("Numeric features    :", bundle.numeric_features)
print("Categorical features:", bundle.categorical_features)
print("Available models    :", available_models())


## 5. Benchmark

`FAST=True` runs a quick 4-model, 3/2-fold pass (good for iteration). Set
`FAST=False` for the full 8-model, 5/3-fold publication run (slower).

In [ ]:
FAST = True  # set False for the full 8-model publication-grade run

if FAST:
    cfg = PipelineConfig(cv=CVConfig(outer_folds=3, inner_folds=2, n_iter_search=8),
                         models=["logistic", "random_forest", "xgboost", "lightgbm"],
                         results_dir=Path("results"))
else:
    cfg = PipelineConfig(cv=CVConfig(outer_folds=5, inner_folds=3, n_iter_search=40),
                         results_dir=Path("results"))

result = run_benchmark(cfg, bundle=bundle, make_plots=True)
result.save(cfg.results_dir)
print("Best model:", result.best_model_name)
show_results(str(cfg.results_dir))


## 6. Permutation importance (model-agnostic)

In [ ]:
imp = feature_importance(result, bundle)
display(imp.head(15))
imp.to_csv("results/feature_importance.csv", index=False)

import matplotlib.pyplot as plt
top = imp.head(12).iloc[::-1]
plt.figure(figsize=(7, 5))
plt.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color="#3b7dd8")
plt.xlabel("Permutation importance (ROC-AUC drop)"); plt.title("Feature importance — best model")
plt.tight_layout(); plt.show()


## 7. Cross-family generalisation (leave-one-material-family-out)

Trains on all-but-one material family and tests on the held-out family — a
direct measure of the cross-family generalisation gap. A large drop vs. the
pooled hold-out AUC in Section 5 quantifies it.

In [ ]:
lofo = leave_one_family_out(cfg, bundle=bundle, model_key="random_forest")
display(lofo)


## 8. For the publication — checklist

1. **Use real data.** Replace Option A with a real eNanoMapper / NIkC / caNanoLab
   export (Options B/C) and map columns to the schema. Join any
   corona-proteomics (PRIDE/MassIVE) or transcriptomic (TG-GATEs/LINCS) features
   you derived offline as extra numeric columns — the pipeline picks them up.
2. **Report nested-CV mean ± std** (Section 5 leaderboard `cv_*` columns), not
   just the single hold-out number — reviewers expect the unbiased estimate.
3. **Report the LOFO table** (Section 7) as your generalisation evidence.
4. **Fix the seed** (`PipelineConfig.random_state`, default 42) and state it.
5. **Save `results/summary.json`** — it records the exact config + per-model
   metrics for the methods section.
6. **State class balance** and prefer **ROC-AUC / PR-AUC / MCC** over accuracy
   (nanotox data is usually imbalanced).

Full repo (package, tests, CLI, roadmap): branch
`claude/nanotoxicology-research-gaps-9vatbv` of your `python-nano-codebase`.
